# PrakashPD — EDA & Feature Engineering

IDBI Innovate 2026, PS4: Default Prediction Model.

**What is real in this notebook:** the structured dataset (UCI "Default of Credit
Card Clients", 30,000 real Taiwanese credit card accounts, Oct 2005 label window),
all EDA statistics, and all engineered features derived from it.

**What is simulated (clearly marked below):** the RM notes/narratives (no such text
exists in the source data), the loan `segment` label and `exposure_at_risk` figure
(the source data is single-product credit-card data, not a multi-segment loan book),
the `loan_vintage_month` used for the out-of-time split (the source data has no date
field at all), and the survival `duration`/`event` columns (the source data has a
single one-month-ahead binary label, not observed multi-month time-to-default).
These simulations exist to demonstrate the modelling *technique* the pitch calls
for (single calibrated cross-segment scale, 12-month horizon via survival analysis,
NLP fusion) on top of a dataset that does not natively contain that structure. See
`README.md` for the full disclosure.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
RNG = np.random.default_rng(42)


## 1. Load raw data

In [2]:
raw = pd.read_excel("../data/raw/default_of_credit_card_clients.xls", header=1)
raw = raw.rename(columns={"default payment next month": "default", "PAY_0": "PAY_1"})
raw = raw.drop(columns=["ID"])
print(raw.shape)
raw.head()


(30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 2. Clean categorical noise

`EDUCATION` and `MARRIAGE` contain undocumented codes (0, 5, 6 for EDUCATION; 0 for MARRIAGE) not in the official codebook. Fold them into an `other` bucket.

In [3]:
df = raw.copy()
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})
df["default"].value_counts(normalize=True)


default
0    0.7788
1    0.2212
Name: proportion, dtype: float64

## 3. EDA

Rare-event base rate, and a look at how repayment-status history relates to default.

In [4]:
default_rate = df["default"].mean()
print(f"Base default rate: {default_rate:.3%}")

pay_cols = [f"PAY_{i}" for i in [1, 2, 3, 4, 5, 6]]
df[pay_cols].describe()


Base default rate: 22.120%


,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,-0.016700,-0.133767,-0.166200,-0.220667,-0.266200,-0.291100
std,1.123802,1.197186,1.196868,1.169139,1.133187,1.149988
min,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000
25%,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000


In [5]:
fig, ax = plt.subplots(figsize=(6, 4))
df.groupby("PAY_1")["default"].mean().plot(kind="bar", ax=ax, color="#1b3a2f")
ax.set_ylabel("default rate")
ax.set_title("Default rate by most-recent repayment status (PAY_1)")
fig.tight_layout()
fig.savefig("../model/eda_default_rate_by_pay1.png", dpi=110)
plt.close(fig)


In [6]:
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
pay_amt_cols = [f"PAY_AMT{i}" for i in range(1, 7)]
num_cols = ["LIMIT_BAL", "AGE"] + pay_cols + bill_cols + pay_amt_cols
corr = df[num_cols + ["default"]].corr()["default"].sort_values(ascending=False)
corr


default      1.000000
PAY_1        0.324794
PAY_2        0.263551
PAY_3        0.235253
PAY_4        0.216614
PAY_5        0.204149
PAY_6        0.186866
AGE          0.013890
BILL_AMT6   -0.005372
BILL_AMT5   -0.006760
BILL_AMT4   -0.010156
BILL_AMT3   -0.014076
BILL_AMT2   -0.014193
BILL_AMT1   -0.019644
PAY_AMT6    -0.053183
PAY_AMT5    -0.055124
PAY_AMT3    -0.056250
PAY_AMT4    -0.056827
PAY_AMT2    -0.058579
PAY_AMT1    -0.072929
LIMIT_BAL   -0.153520
Name: default, dtype: float64

## 4. Feature engineering (real, derived from the source fields)

In [7]:
eng = df.copy()

eng["util_ratio_1"] = (eng["BILL_AMT1"] / eng["LIMIT_BAL"].replace(0, np.nan)).clip(-2, 5).fillna(0)
eng["avg_util_ratio"] = (eng[bill_cols].mean(axis=1) / eng["LIMIT_BAL"].replace(0, np.nan)).clip(-2, 5).fillna(0)

pay_ratio = eng[pay_amt_cols].to_numpy() / np.where(eng[bill_cols].to_numpy() == 0, np.nan, eng[bill_cols].to_numpy())
eng["avg_pay_ratio"] = np.nan_to_num(np.nanmean(pay_ratio, axis=1), nan=0.0, posinf=5.0, neginf=-5.0)

eng["months_delinquent"] = (eng[pay_cols] > 0).sum(axis=1)
eng["max_delinquency"] = eng[pay_cols].max(axis=1)
eng["delinquency_trend"] = eng["PAY_1"] - eng["PAY_6"]
eng["bill_trend"] = eng["BILL_AMT1"] - eng["BILL_AMT6"]

eng[["util_ratio_1", "avg_util_ratio", "avg_pay_ratio", "months_delinquent",
     "max_delinquency", "delinquency_trend", "bill_trend"]].describe()


C:\Users\asus\AppData\Local\Temp\ipykernel_1104\2398580759.py:7: RuntimeWarning: Mean of empty slice
  eng["avg_pay_ratio"] = np.nan_to_num(np.nanmean(pay_ratio, axis=1), nan=0.0, posinf=5.0, neginf=-5.0)


,util_ratio_1,avg_util_ratio,avg_pay_ratio,months_delinquent,max_delinquency,delinquency_trend,bill_trend
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,0.423713,0.373036,-2.209925,0.834200,0.438733,0.274400,12351.570500
std,0.410718,0.351724,116.656430,1.554303,1.345154,1.165683,43922.421534
min,-0.619892,-0.232590,-13727.477930,0.000000,-2.000000,-7.000000,-428791.000000
25%,0.022032,0.029997,0.038134,0.000000,0.000000,0.000000,-2963.000000
50%,0.313994,0.284834,0.066987,0.000000,0.000000,0.000000,923.000000
75%,0.829843,0.687929,0.708112,1.000000,2.000000,1.000000,19793.750000
max,5.000000,5.000000,4444.133333,6.000000,8.000000,7.000000,708323.000000


## 5. Simulated loan segment + exposure at risk

**Simulated.** The source dataset is single-product (credit card). To demo a cross-segment watchlist, we assign a synthetic `segment` label, weighted so higher `LIMIT_BAL` skews toward SME/commercial-shaped segments, and derive `exposure_at_risk` from the real `LIMIT_BAL` and utilization fields (not fabricated from nothing, but the segment taxonomy itself is invented for the demo).

In [8]:
segments = np.array(["Retail Personal", "Credit Card", "Consumer Durable", "SME", "Agri"])
limit_bin = pd.qcut(eng["LIMIT_BAL"], 5, labels=False, duplicates="drop")
segment_weights = {
    0: [0.45, 0.35, 0.15, 0.03, 0.02],
    1: [0.35, 0.30, 0.20, 0.10, 0.05],
    2: [0.20, 0.25, 0.20, 0.25, 0.10],
    3: [0.10, 0.15, 0.15, 0.45, 0.15],
    4: [0.05, 0.10, 0.10, 0.55, 0.20],
}
eng["segment"] = [
    RNG.choice(segments, p=segment_weights.get(int(b), [0.2, 0.2, 0.2, 0.2, 0.2]))
    for b in limit_bin.fillna(0)
]
eng["exposure_at_risk"] = (eng["LIMIT_BAL"] * (0.3 + eng["avg_util_ratio"].clip(0, 1.5))).round(2)
eng["segment"].value_counts(normalize=True)


segment
SME                 0.266033
Retail Personal     0.239767
Credit Card         0.233933
Consumer Durable    0.159600
Agri                0.100667
Name: proportion, dtype: float64

## 6. Simulated RM notes / narratives

**Simulated.** No free-text exists in the source data. We generate short template-based English notes per borrower whose sentiment and content are correlated with the borrower's real delinquency trend (not random noise), so the NLP feature fusion step downstream has genuine signal to pick up rather than a decorative feature that does nothing.

In [9]:
NEGATIVE_TEMPLATES = [
    "Borrower reports {n} consecutive late payments this quarter.",
    "RM flags declining GST turnover, down {pct}% over two quarters.",
    "Client mentioned cash flow stress and requested a payment holiday.",
    "Repeated bounced auto-debit on the linked account this cycle.",
    "RM notes irregular business income and rising informal borrowing.",
    "Borrower's shop has seen a sharp drop in footfall, sales down noticeably.",
    "Spoke to client, admits taking a loan from a moneylender to cover EMI.",
    "Household income has dropped after a job loss in the family.",
    "Business partner exited unexpectedly, borrower now managing alone with strain.",
    "Client requested a moratorium citing a medical emergency in the family.",
    "Seasonal slowdown has hit revenue hard this quarter, borrower sounded worried.",
    "RM visit found the shop shuttered for renovation with no clear reopening date.",
    "Borrower mentioned a major client delayed payment, squeezing working capital.",
    "Supplier raised prices sharply, borrower struggling to maintain margins.",
    "Client's other loan account was recently flagged for default by another lender.",
    "Rent on the business premises was hiked, borrower says it is unsustainable.",
    "Borrower confided that a family dispute is affecting the business finances.",
    "Local competition has intensified, borrower reports falling order volumes.",
    "Bank statement shows multiple small-value withdrawals suggesting distress borrowing.",
    "Borrower missed the scheduled RM call twice this month without explanation.",
]
NEUTRAL_TEMPLATES = [
    "Routine review call completed, no material change flagged.",
    "Borrower confirmed employment status unchanged.",
    "Standard KYC refresh completed this month.",
    "No response yet to the scheduled review call, will follow up next cycle.",
    "Business operations continuing as usual per borrower's own account.",
    "Borrower relocated business premises within the same city, no other change.",
    "Annual documentation update completed without incident.",
    "Borrower requested a statement copy for tax filing purposes.",
]
POSITIVE_TEMPLATES = [
    "Borrower reports steady GST turnover growth of {pct}% this quarter.",
    "Client cleared prior dues ahead of schedule.",
    "RM notes new stable income source, salary credit regularized.",
    "Business expansion noted, turnover trending up.",
    "Borrower just signed a large new export order, expects revenue to grow sharply.",
    "Client paid off an old personal loan completely, now debt free elsewhere.",
    "Got a promotion at work with a solid raise starting next month.",
    "Borrower opened a second outlet in a nearby market, business is scaling well.",
    "RM notes healthy cash reserves being built up over recent months.",
    "Client's spouse started a stable job, household income has improved.",
    "Long-term contract renewed with the borrower's largest customer.",
    "Borrower reports strong festive season sales, well above last year.",
    "New equipment purchased outright without needing additional financing.",
    "Borrower has started prepaying the loan ahead of the due schedule.",
    "Local demand has picked up strongly, borrower sounded confident on the call.",
    "Borrower's family business received a government subsidy, easing costs.",
    "Client's credit profile improved enough to get a private bank overdraft.",
    "Referral business has picked up, borrower reports a healthy new customer pipeline.",
]

def make_note(row):
    trend = row["delinquency_trend"]
    months_delinq = row["months_delinquent"]
    if trend > 0 or months_delinq >= 3:
        tmpl = RNG.choice(NEGATIVE_TEMPLATES)
        return tmpl.format(n=int(max(months_delinq, 1)), pct=int(RNG.integers(15, 45)))
    if trend < 0 and months_delinq == 0:
        tmpl = RNG.choice(POSITIVE_TEMPLATES)
        return tmpl.format(pct=int(RNG.integers(5, 25)))
    return RNG.choice(NEUTRAL_TEMPLATES)

eng["rm_note"] = eng.apply(make_note, axis=1)
eng[["delinquency_trend", "months_delinquent", "rm_note"]].sample(5, random_state=1)


,delinquency_trend,months_delinquent,rm_note
10747,0,0,"No response yet to the scheduled review call, ..."
12573,-3,2,Business operations continuing as usual per bo...
29676,0,0,Annual documentation update completed without ...
8856,0,6,Seasonal slowdown has hit revenue hard this qu...
21098,1,3,Client mentioned cash flow stress and requeste...


## 7. Simulated survival duration/event (12-month-horizon proxy)

**Simulated / approximated.** The source label is a single one-month-ahead binary flag — there is no genuine observed multi-month time-to-default. To demonstrate a survival-analysis component at all, we derive a `duration` (months of observed history, 1-6) and `event` from the *real* repayment-status trend: borrowers whose delinquency escalates are treated as the event occurring at the month it first crossed into delinquency; everyone else is right-censored at 6 months. This is a reasonable technique demo, not a validated 12-month time-to-default estimate — flagged plainly in the README.

In [10]:
def first_delinquent_month(row):
    for i, col in enumerate([f"PAY_{k}" for k in [1, 2, 3, 4, 5, 6]], start=1):
        if row[col] > 0:
            return i
    return 6

eng["duration"] = eng.apply(first_delinquent_month, axis=1)
eng["event"] = eng["default"]
eng[["duration", "event"]].describe()


,duration,event
count,30000.000000,30000.000000
mean,4.625767,0.221200
std,2.119497,0.415062
min,1.000000,0.000000
25%,3.000000,0.000000
50%,6.000000,0.000000
75%,6.000000,0.000000
max,6.000000,1.000000


## 8. Simulated loan vintage + out-of-time split

**Simulated.** The source data has no date field whatsoever. To honor the brief's requirement for a *time-based* train/test split (rather than a random split, which would overstate performance for a rare-event target), we assign each row a synthetic `loan_vintage_month` (1-12) at random and treat months 1-9 as train and 10-12 as held-out. This demonstrates the *methodology* the brief asks for; it is not a real temporal validation since the underlying rows have no true chronology. Treat the resulting metrics as illustrative of the OOT methodology, not a certified forward-time backtest.

In [11]:
eng["loan_vintage_month"] = RNG.integers(1, 13, size=len(eng))
train = eng[eng["loan_vintage_month"] <= 9].reset_index(drop=True)
test = eng[eng["loan_vintage_month"] > 9].reset_index(drop=True)
print("train:", train.shape, "test:", test.shape)
print("train default rate:", train["default"].mean(), "test default rate:", test["default"].mean())


train: (22434, 37) test: (7566, 37)
train default rate: 0.2214941606490149 test default rate: 0.22032778218345228


## 9. Save processed data

In [12]:
import os
os.makedirs("../data/processed", exist_ok=True)
eng.to_csv("../data/processed/full.csv", index=False)
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)
print("saved.")


saved.
